# 方法2最小实验：Block/Site 级 Memorization Patching

本 notebook 目标：用仓库现有接口，快速跑通一个 **block/site 级** 的 clean vs corrupted vs patched/ablated 对照实验。

范围刻意最小：
- 只做方法2（Model Attribution）
- 不做 head-level / neuron-level
- 不重写训练框架


In [ ]:
import json
import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt

# 定位仓库根目录（兼容从仓库根或 notebook 子目录启动）
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / "mini_gpt_attnres").exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError("未找到仓库根目录（应包含 mini_gpt_attnres 目录）")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)


## Checkpoint 选择策略

本 notebook **只使用你指定的两份训练完成 checkpoint**（不加载额外 GPT-2 模型）：
- standard: `/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/standard/ckpt_standard_last.pt`
- attnres: `/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/attnres/ckpt_attnres_last.pt`

通过 `MODEL_VARIANT` 在两者间切换（`"standard"` 或 `"attnres"`）。
当前 notebook 默认使用 `standard`。


In [ ]:
from mini_gpt_attnres.evaluate import load_checkpoint

CHECKPOINTS = {
    "standard": "/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/standard/ckpt_standard_last.pt",
    "attnres": "/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/attnres/ckpt_attnres_last.pt",
}

# 在这里切换模型："standard" / "attnres"
MODEL_VARIANT = "standard"
if MODEL_VARIANT not in CHECKPOINTS:
    raise ValueError(f"MODEL_VARIANT 必须是 {list(CHECKPOINTS.keys())}")

ckpt_path = Path(CHECKPOINTS[MODEL_VARIANT])

model, experiment, checkpoint = load_checkpoint(ckpt_path, device=torch.device("cpu"))
model = model.to(torch.device("cpu")).eval()
device = torch.device("cpu")

print("model_variant:", MODEL_VARIANT)
print("checkpoint:", ckpt_path)
print("model_type:", checkpoint.get("model_type"))
print("step:", checkpoint.get("step"))
print("n_layer:", experiment.model.n_layer)
print("n_head:", experiment.model.n_head)
print("n_embd:", experiment.model.n_embd)
print("vocab_size:", experiment.model.vocab_size)
print("block_size:", experiment.model.block_size)
print("device:", next(model.parameters()).device)


## 准备小样本输入（clean vs corrupted）

这里直接基于 checkpoint 的 `vocab_size` 构造 token id 序列：
- 不调用 `transformers` tokenizer
- 不加载任何额外 GPT-2 权重
- 仅依赖当前 checkpoint 恢复出的模型


In [ ]:
from mini_gpt_attnres.interp.memorization_runner import MemorizationPatchingRunner

runner = MemorizationPatchingRunner(model)

# 仅基于当前 checkpoint 的词表构造最小样本，不加载 GPT-2/tokenizer。
seq_len = int(min(8, experiment.model.block_size))
if seq_len < 4:
    raise ValueError(f"block_size={experiment.model.block_size} 过小，至少需要 >=4")
if experiment.model.vocab_size < 2:
    raise ValueError(f"vocab_size={experiment.model.vocab_size} 过小，至少需要 >=2")

torch.manual_seed(0)
base_tokens = torch.randint(
    low=0,
    high=experiment.model.vocab_size,
    size=(1, seq_len),
    dtype=torch.long,
    device=device,
)

clean_tokens = base_tokens.clone()
corrupted_tokens = base_tokens.clone()
corrupt_position = min(2, seq_len - 2)
corrupted_tokens[0, corrupt_position] = (corrupted_tokens[0, corrupt_position] + 1) % experiment.model.vocab_size

sample_info = {
    "source": "synthetic_token_ids",
    "vocab_size": int(experiment.model.vocab_size),
    "seq_len": int(seq_len),
    "corrupt_position": int(corrupt_position),
    "note": "只基于训练 checkpoint 构造样本，不加载外部 GPT-2/tokenizer",
}

print("sample_info:", sample_info)
print("clean_tokens:", clean_tokens.tolist())
print("corrupted_tokens:", corrupted_tokens.tolist())

target_position = clean_tokens.size(1) - 1
target_token_id = int(clean_tokens[0, target_position].item())
print("target_position:", target_position, "target_token_id:", target_token_id)


## Clean forward + cache 可观测位点

这里明确列出 cache keys，确定可用于 block/site 级 patching 的位点。

In [ ]:
from mini_gpt_attnres.interp.analysis_adapter import AnalysisAdapter

with torch.no_grad():
    clean_outputs = model(clean_tokens, return_intermediates=True, return_attn=True, return_cache=True)

cache = clean_outputs["cache"]
cache_keys = sorted(cache.data.keys())

print("forward output keys:", sorted(clean_outputs.keys()))
print("cache key count:", len(cache_keys))
print("\n前 30 个 cache keys:")
for k in cache_keys[:30]:
    print(" ", k)

candidate_sites = [k for k in cache_keys if k.endswith(("attn_out", "mlp_out", "output"))]
print("\n可用于 block/site patching 的候选位点（前 20 个）:")
for k in candidate_sites[:20]:
    print(" ", k)

trace = AnalysisAdapter.from_model_outputs(clean_outputs)
print("\ntrace layers:", list(trace.layers()))
print("layer0 block_type:", trace.layer(0).block_type)


## 选择 patch site

首轮实验选 `blocks.0.attn_out`：
- 位点稳定存在于当前模型 cache
- 属于标准 block 级 attention 输出
- 便于解释 clean->corrupted->patched 的差异

先做 block/site 级，避免 head-level 额外不稳定因素。

In [ ]:
from mini_gpt_attnres.interp.ablation import zero_ablation_override

patch_site = "blocks.0.attn_out"
if patch_site not in cache.data:
    raise KeyError(f"patch_site 不在 cache 中: {patch_site}")

# 1) 使用现有 MemorizationPatchingRunner 做 clean->corrupted patch
result_patch = runner.run(
    clean_input=clean_tokens,
    corrupted_input=corrupted_tokens,
    patch_site=patch_site,
    target_position=target_position,
    target_token_id=target_token_id,
    metric="logprob",
)

# 2) 使用现有 ablation helper 做 zero ablation
with torch.no_grad():
    corrupted_outputs = model(corrupted_tokens, return_intermediates=True, return_cache=True)
    ablated_outputs = model(
        corrupted_tokens,
        return_intermediates=True,
        return_cache=True,
        activation_overrides=zero_ablation_override(patch_site),
    )

def token_logprob(outputs, pos, token_id):
    logits = outputs["logits"]
    return float(torch.log_softmax(logits, dim=-1)[0, pos, token_id].item())

clean_lp = float(result_patch.baseline_clean_score.item())
corr_lp = float(result_patch.baseline_corrupted_score.item())
patch_lp = float(result_patch.patched_score.item())
ablate_lp = token_logprob(ablated_outputs, target_position, target_token_id)

print("patch_site:", patch_site)
print("clean logprob:", clean_lp)
print("corrupted logprob:", corr_lp)
print("patched logprob:", patch_lp)
print("zero-ablated logprob:", ablate_lp)
print("patch effect (patched - corrupted):", float(result_patch.effect_size.item()))


## 最小指标：目标 token 分数与 top-k 变化

这里用最简单稳定的指标：
- 指定位置/目标 token 的 logprob
- top-k 概率分布变化


In [ ]:
def topk_tokens(outputs, pos, k=5):
    logits = outputs["logits"][0, pos]
    probs = torch.softmax(logits, dim=-1)
    vals, idx = torch.topk(probs, k=k)
    return [(int(i), float(v)) for i, v in zip(idx, vals)]

with torch.no_grad():
    clean_outputs_eval = model(clean_tokens)
    corr_outputs_eval = model(corrupted_tokens)
    patched_outputs_eval = model(
        corrupted_tokens,
        activation_overrides={patch_site: clean_outputs["cache"][patch_site].clone()},
    )

rows = [
    {"condition": "clean", "target_logprob": clean_lp, "top5": topk_tokens(clean_outputs_eval, target_position)},
    {"condition": "corrupted", "target_logprob": corr_lp, "top5": topk_tokens(corr_outputs_eval, target_position)},
    {"condition": "patched_from_clean", "target_logprob": patch_lp, "top5": topk_tokens(patched_outputs_eval, target_position)},
    {"condition": "zero_ablated", "target_logprob": ablate_lp, "top5": topk_tokens(ablated_outputs, target_position)},
]

print(json.dumps(rows, indent=2, ensure_ascii=False))


## 可视化（最小版）

左图：目标 token logprob 对照（clean/corrupted/patched/ablated）
右图：多个 block/site 的 patch effect（patched - corrupted）


In [ ]:
sweep_sites = [
    "blocks.0.attn_out",
    "blocks.0.mlp_out",
    "blocks.1.attn_out",
    "blocks.1.mlp_out",
]
sweep_sites = [s for s in sweep_sites if s in cache.data]

sweep = runner.run_site_sweep(
    clean_input=clean_tokens,
    corrupted_input=corrupted_tokens,
    patch_sites=sweep_sites,
    target_position=target_position,
    target_token_id=target_token_id,
    metric="logprob",
)
site_effects = [float(v.item()) for v in sweep.effect_by_site[:, 0]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

conditions = ["clean", "corrupted", "patched", "ablated"]
values = [clean_lp, corr_lp, patch_lp, ablate_lp]
axes[0].bar(conditions, values)
axes[0].set_title(f"Target logprob @pos={target_position}, token={target_token_id}")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(range(len(sweep_sites)), site_effects)
axes[1].set_xticks(range(len(sweep_sites)))
axes[1].set_xticklabels(sweep_sites, rotation=30, ha="right")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_title("Patch effect by site")

plt.tight_layout()
plt.show()

print("site_effects:")
for s, e in zip(sweep_sites, site_effects):
    print(f"{s:20s} {e:+.6f}")


## 结果解释（最小结论）

这份 notebook 实际完成了：
1. 从真实可加载 checkpoint 恢复模型并 `eval` 推理
2. 构造 clean/corrupted 输入
3. 通过现有 cache 链路识别可 patch 的 block/site
4. 用现有 `MemorizationPatchingRunner` 做 block/site patching
5. 用现有 `zero_ablation_override` 做同位点消融
6. 对比目标 token 分数和多 site effect

它属于方法2（Model Attribution）的原因：
- 干预发生在 transformer block/site 层级
- 直接比较干预前后输出分数变化
- 可以扩展到 layer sweep

当前结果能说明：
- 在当前样本和位点上，某些 block/site 对目标 token 分数有可测影响。

当前结果还不能说明：
- 论文级普适因果结论（样本量、任务协议、统计检验都还不够）。

补充：本实验输入样本为 checkpoint 词表上的合成 token id，不加载额外 GPT-2 模型或 tokenizer。
